# SakThai 1.5B v2 — Google Colab Training

QLoRA + rsLoRA + all-linear targets + completion-only loss.
Fine-tunes Qwen2.5-1.5B-Instruct for tool-calling on the SakThai v7 dataset.

**Expected time on T4:** ~2-3 hours

---
### Setup
Set your HF token as a **Colab Secret**:
1. Click the 🔑 key icon in the left sidebar → **Add new secret**
2. Name: `HF_TOKEN`, Value: your HF write token
3. Toggle the notebook access switch ON
4. Then run all cells below

In [ ]:
# Install deps
!pip install -qU "transformers>=4.44" "trl>=0.19,<0.20" peft datasets accelerate bitsandbytes huggingface_hub

In [ ]:
import os, torch
from google.colab import userdata
from datasets import load_dataset, concatenate_datasets
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer
from huggingface_hub import login

HF_TOKEN = userdata.get('HF_TOKEN')
assert HF_TOKEN, "Set HF_TOKEN in Colab secrets (🔑 sidebar)"
login(token=HF_TOKEN)

BASE_MODEL   = "Qwen/Qwen2.5-1.5B-Instruct"
HF_USER      = "Nanthasit"
ADAPTER_REPO = f"{HF_USER}/sakthai-context-1.5b-tools-v2"
MERGED_REPO  = f"{HF_USER}/sakthai-context-1.5b-merged-v2"
MAX_SEQ_LEN  = 2048
print(f"Authenticated as {HF_USER}")

In [ ]:
bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=bnb, device_map="auto", torch_dtype=torch.bfloat16,
)
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model.config.use_cache = False

lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    use_rslora=True,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
def to_text(ex):
    msgs = ex["messages"]
    tools = ex.get("tools") or None
    return {"text": tokenizer.apply_chat_template(
        msgs, tools=tools, tokenize=False, add_generation_prompt=False,
    )}

main = load_dataset(f"{HF_USER}/sakthai-combined-v7", split="train")
train_data = main.map(to_text, remove_columns=main.column_names)
try:
    supp = load_dataset(f"{HF_USER}/sakthai-irrelevance-supplement", split="train")
    supp_text = supp.map(to_text, remove_columns=supp.column_names)
    train_data = concatenate_datasets([train_data, supp_text])
except Exception as e:
    print("supplement skipped:", e)

eval_raw = load_dataset(f"{HF_USER}/sakthai-combined-v7", split="test")
eval_data = eval_raw.map(to_text, remove_columns=eval_raw.column_names)
print(f"train={len(train_data)}  eval={len(eval_data)}")

In [ ]:
args = SFTConfig(
    output_dir="./sakthai-1.5b-lora",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    optim="adamw_8bit",
    learning_rate=2e-4, lr_scheduler_type="cosine", warmup_ratio=0.03,
    logging_steps=10, eval_strategy="steps", eval_steps=50,
    save_strategy="steps", save_steps=100, save_total_limit=2,
    load_best_model_at_end=True, metric_for_best_model="eval_loss",
    bf16=True, tf32=True, report_to="none",
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    completion_only_loss=True,
    push_to_hub=True,
    hub_model_id=ADAPTER_REPO,
    hub_strategy="every_save",
)
trainer = SFTTrainer(
    model=model, processing_class=tokenizer, args=args,
    train_dataset=train_data, eval_dataset=eval_data,
)
trainer.train()

In [ ]:
# Push adapter
trainer.save_model("./sakthai-1.5b-lora-best")
tokenizer.save_pretrained("./sakthai-1.5b-lora-best")
trainer.model.push_to_hub(ADAPTER_REPO)
tokenizer.push_to_hub(ADAPTER_REPO)
print(f"Adapter pushed: {ADAPTER_REPO}")

In [ ]:
# Merge adapter into full weights and push merged model
from peft import PeftModel
del model, trainer
torch.cuda.empty_cache()

base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=torch.bfloat16, device_map="auto",
)
merged = PeftModel.from_pretrained(base, "./sakthai-1.5b-lora-best").merge_and_unload()
merged.push_to_hub(MERGED_REPO)
tokenizer.push_to_hub(MERGED_REPO)
print(f"Merged pushed: {MERGED_REPO}")
print("\n✅ Done!")